In [ ]:
import pandas as pd
import os

RAW_DIR = "../data/01_raw/"
INTERMEDIATE_DIR = "../data/02_intermediate/"
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

pks_de_path = os.path.join(RAW_DIR, "pks_deutsche_roh.xlsx")
pks_nde_path = os.path.join(RAW_DIR, "pks_nichtdeutsche_roh.xlsx")

def load_and_clean_pks(filepath, count_col_name):
    """Lädt eine BKA-PKS Excel-Datei, findet den Header dynamisch und bereinigt die Daten."""
    print(f"Verarbeite: {filepath} ...")
    
    # Excel roh laden um Header-Zeile zu finden (alles als String, um Formatierungsfehler zu vermeiden)
    df_raw = pd.read_excel(filepath, header=None, dtype=str)
    
    # Header beginnt bei der Zeile mit "Schlüssel"
    try:
        start_row = df_raw[df_raw[0] == "Schlüssel"].index[0]
    except IndexError:
        print(f"Warnung: Konnte 'Schlüssel' nicht finden. Versuche Zeile 4.")
        start_row = 4
        
    df = pd.read_excel(filepath, header=start_row)
    
    # Spalten umbenennen BKA-Formatierung variiert zwischen Jahren,
    # daher flexible Suche nach der Tatverdächtigen-Spalte
    suspects_col = [c for c in df.columns if "Tatver" in str(c) and "insgesamt" in str(c)]
    if suspects_col:
        df = df.rename(columns={suspects_col[0]: count_col_name})
    
    df = df.rename(columns={
        "Schlüssel": "Deliktschlüssel",
        "Straftat": "Straftat",
        "Jahr": "Jahr"
    })
    
    # Nur relevante Spalten behalten, Fußzeilen/NaN-Zeilen rauswerfen
    cols_to_keep = ["Jahr", "Deliktschlüssel", "Straftat", count_col_name]
    df = df[[c for c in cols_to_keep if c in df.columns]]
    df = df.dropna(subset=['Deliktschlüssel', 'Jahr'])
    
    # Non-Breaking-Spaces (\xa0) und Mehrfach-Leerzeichen bereinigen
    df['Straftat'] = df['Straftat'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # Deliktschlüssel vereinheitlichen: Excel liest manche Codes als int, andere als str
    # -> ohne Vereinheitlichung schlägt der Inner Join fehl (z.B. 220000 != '220000')
    df['Deliktschlüssel'] = df['Deliktschlüssel'].astype(str)
    
    df["Jahr"] = df["Jahr"].astype(float).astype(int)
    if count_col_name in df.columns:
        df[count_col_name] = df[count_col_name].astype(float).astype(int)
        
    return df

# Beide Dateien getrennt laden und bereinigen
df_deutsch = load_and_clean_pks(pks_de_path, "anzahl_deutsch")
df_nichtdeutsch = load_and_clean_pks(pks_nde_path, "anzahl_nichtdeutsch")

# Inner Join über die gemeinsamen Schlüssel
df_pks = pd.merge(
    df_deutsch, 
    df_nichtdeutsch, 
    on=["Jahr", "Deliktschlüssel", "Straftat"], 
    how="inner"
)

df_pks["anzahl_gesamt"] = df_pks["anzahl_deutsch"] + df_pks["anzahl_nichtdeutsch"]

output_path = os.path.join(INTERMEDIATE_DIR, "pks_cleaned.csv")
df_pks.to_csv(output_path, index=False, sep=";")

print(f"\nErfolg! {len(df_pks)} Zeilen kombiniert. Gespeichert unter: {output_path}")
display(df_pks.head())